# Square-Dalitz SCF migration

SCF remains a specialized detector-resolution feature, so this notebook keeps the dedicated `SquareDalitzSCFMap`/`SCFSignalPDF` objects. The amplitude toy and plots use the high-level helpers.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel,DecayModel,NonResonant,RealImag,SCFSignalPDF,
    SquareDalitzSCFMap,enable_x64,generate_toy,plot_square_dalitz,
)
from dalitzplotfitter.integration import GridIntegrator
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=120,
    normalization_pair=(0,2),
)
N=18
nbin=N*N
migration=np.eye(nbin)*0.65
migration+=np.roll(np.eye(nbin),1,axis=1)*0.175
migration+=np.roll(np.eye(nbin),-1,axis=1)*0.175
migration/=migration.sum(axis=1,keepdims=True)
fraction=np.full(nbin,0.15)

scf_map=SquareDalitzSCFMap(
    migration,fraction,model.channel.parent_mass,model.channel.daughter_masses,
    N,N,pair=(0,2),
)
pdf=SCFSignalPDF(
    intensity=lambda d,p:model.intensity(d,p),
    integrator=GridIntegrator(model.normalization_sample),
    scf_map=scf_map,
)


In [ ]:
toy=generate_toy(model,20_000,seed=707,pool_size=120_000)
plot_square_dalitz(
    toy,mother_mass=model.channel.parent_mass,masses=model.channel.daughter_masses,
    pair=(0,2),title="Underlying generated signal"
)
plt.show()

true_density=model.intensity(scf_map.true_bin_data(),{})
reco_density=scf_map.smeared_bin_density(true_density)
plt.figure(figsize=(6,5))
plt.imshow(np.asarray(reco_density).reshape(N,N).T,origin="lower",aspect="auto")
plt.xlabel("$m'$ bin")
plt.ylabel(r"$\theta'$ bin")
plt.title("Migrated SCF density")
plt.colorbar()
plt.show()
print("SCF PDF normalization:",float(pdf.normalization({})))
